# FACTS Sea-Level Dashboard Generator

This notebook generates an interactive HTML dashboard from [FACTS](https://github.com/radical-collaboration/facts) sea-level projection output files.

**No installation needed beyond running these cells.**

---

### How to use
1. Run **Cell 1** to install dependencies (takes ~1 minute, only needed once per session)
2. Run **Cell 2** to connect your Google Drive
3. Paste your Drive folder link into **Cell 3** and run it
4. Run **Cell 4** to generate the dashboard
5. Run **Cell 5** to download the HTML file to your computer

> **Tip:** Run a cell by clicking it and pressing `Shift + Enter`, or click the play button on the left.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run this once at the start of each session. Takes about 1 minute.

print('Installing dependencies...')
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'bokeh>=3.4', 'xarray>=2023.1', 'numpy>=1.24',
                'pandas>=2.0', 'netCDF4>=1.6', '--quiet'], check=True)

import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/Ttheegela/facts.plotting.dashboard/main/facts_dashboard.py',
    'facts_dashboard.py'
)

print('Done! Ready to generate dashboards.')

In [ ]:
# ── Cell 2: Connect Google Drive ─────────────────────────────────────────────
# This lets the notebook access your FACTS data stored in Google Drive.
# Click "Connect to Google Drive" when the popup appears.

from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected.')

In [ ]:
# ── Cell 3: Set your data folder ──────────────────────────────────────────────
# 1. Go to Google Drive and find your FACTS experiment folder
# 2. Right-click the folder and select "Copy link"
# 3. Paste the link below

DRIVE_LINK = "https://drive.google.com/drive/folders/YOUR_FOLDER_ID"   # <-- PASTE LINK HERE

TITLE  = "FACTS Sea-Level Projections"   # Optional: change the dashboard title
OUTPUT = "/content/facts_dashboard.html"

# ── Resolve the link to a folder path ────────────────────────────────────────
import re, os
from google.auth import default
from googleapiclient.discovery import build

match = re.search(r'/folders/([a-zA-Z0-9_-]+)', DRIVE_LINK)
if not match:
    print('ERROR: That does not look like a Google Drive folder link.')
    print('Right-click your folder in Drive and select "Copy link".')
else:
    folder_id = match.group(1)
    creds, _ = default()
    svc = build('drive', 'v3', credentials=creds)

    def _resolve_path(svc, fid):
        parts = []
        cur = fid
        while cur:
            meta = svc.files().get(fileId=cur, fields='name,parents', supportsAllDrives=True).execute()
            parts.append(meta['name'])
            parents = meta.get('parents', [])
            cur = parents[0] if parents else None
        parts.reverse()
        inner = '/'.join(parts[1:])   # strip root node
        if parts and parts[0] != 'My Drive':
            return '/content/drive/Shareddrives/' + inner
        return '/content/drive/MyDrive/' + inner

    EXP_ROOT = _resolve_path(svc, folder_id)

    if os.path.isdir(EXP_ROOT):
        contents = os.listdir(EXP_ROOT)
        print(f'Found folder: {EXP_ROOT}')
        print(f'{len(contents)} items:')
        for item in sorted(contents)[:10]:
            print(f'  {item}')
        if len(contents) > 10:
            print(f'  ... and {len(contents)-10} more')
        print('\nPath looks good. Run Cell 4 to generate the dashboard.')
    else:
        print(f'ERROR: Folder not found at: {EXP_ROOT}')
        print('If this folder was shared with you, right-click it in Drive')
        print('and select "Add shortcut to Drive", then re-run Cell 2 and Cell 3.')

In [ ]:
# ── Cell 4: Generate the dashboard ───────────────────────────────────────────
# This may take 30-60 seconds depending on the size of your dataset.

import subprocess, sys

cmd = [sys.executable, 'facts_dashboard.py',
       '--exp-root', EXP_ROOT,
       '--output',   OUTPUT,
       '--title',    TITLE]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

if result.returncode == 0:
    size_mb = os.path.getsize(OUTPUT) / (1024 * 1024)
    print(f'Dashboard ready ({size_mb:.1f} MB). Run Cell 5 to download it.')
else:
    print('Something went wrong:')
    print(result.stderr)

In [ ]:
# ── Cell 5: Download the dashboard ───────────────────────────────────────────
# Downloads facts_dashboard.html to your computer.
# Open the file in any browser -- no internet connection needed.

from google.colab import files
files.download(OUTPUT)
print('Download started. Check your browser downloads folder.')

---

## Other input modes

The cells above handle the standard case (multiple SSP scenarios under one folder).
For other data layouts, use the cells below instead of Cell 3 and 4.

---

In [ ]:
# ── Alternative: Single NetCDF file ──────────────────────────────────────────
# Use this if you have one .nc file (e.g. a total sea-level file).
# Right-click the file in Google Drive and select "Copy link", then paste below.

DRIVE_LINK = "https://drive.google.com/file/d/YOUR_FILE_ID/view"   # <-- PASTE LINK HERE
OUTPUT     = "/content/facts_dashboard.html"

import re, os, subprocess, sys
from google.auth import default
from googleapiclient.discovery import build

match = re.search(r'/file/d/([a-zA-Z0-9_-]+)', DRIVE_LINK)
if not match:
    print('ERROR: That does not look like a Google Drive file link.')
else:
    file_id = match.group(1)
    creds, _ = default()
    svc = build('drive', 'v3', credentials=creds)

    def _resolve_file_path(svc, fid):
        parts = []
        cur = fid
        while cur:
            meta = svc.files().get(fileId=cur, fields='name,parents', supportsAllDrives=True).execute()
            parts.append(meta['name'])
            parents = meta.get('parents', [])
            cur = parents[0] if parents else None
        parts.reverse()
        inner = '/'.join(parts[1:])
        if parts and parts[0] != 'My Drive':
            return '/content/drive/Shareddrives/' + inner
        return '/content/drive/MyDrive/' + inner

    SINGLE_FILE = _resolve_file_path(svc, file_id)
    print(f'Found file: {SINGLE_FILE}')

    result = subprocess.run(
        [sys.executable, 'facts_dashboard.py', '--single-nc-file', SINGLE_FILE, '--output', OUTPUT],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)

In [ ]:
# ── Alternative: AR6-style confidence files ───────────────────────────────────
# Use this if you have a folder with medium_confidence/ and low_confidence/ subfolders.
# Right-click the top-level confidence folder in Drive and select "Copy link".

DRIVE_LINK = "https://drive.google.com/drive/folders/YOUR_FOLDER_ID"   # <-- PASTE LINK HERE
CONF_LEVEL = "both"    # Options: "both", "medium_confidence", "low_confidence"
OUTPUT     = "/content/facts_dashboard_confidence.html"

import re, os, subprocess, sys
from google.auth import default
from googleapiclient.discovery import build

match = re.search(r'/folders/([a-zA-Z0-9_-]+)', DRIVE_LINK)
if not match:
    print('ERROR: That does not look like a Google Drive folder link.')
else:
    folder_id = match.group(1)
    creds, _ = default()
    svc = build('drive', 'v3', credentials=creds)

    def _resolve_path(svc, fid):
        parts = []
        cur = fid
        while cur:
            meta = svc.files().get(fileId=cur, fields='name,parents', supportsAllDrives=True).execute()
            parts.append(meta['name'])
            parents = meta.get('parents', [])
            cur = parents[0] if parents else None
        parts.reverse()
        inner = '/'.join(parts[1:])
        if parts and parts[0] != 'My Drive':
            return '/content/drive/Shareddrives/' + inner
        return '/content/drive/MyDrive/' + inner

    CONF_ROOT = _resolve_path(svc, folder_id)
    print(f'Found folder: {CONF_ROOT}')

    result = subprocess.run(
        [sys.executable, 'facts_dashboard.py',
         '--confidence-root',  CONF_ROOT,
         '--confidence-level', CONF_LEVEL,
         '--output',           OUTPUT],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)